# Brady PoroTomo DTS — EFGP benchmark

Distributed temperature sensing (DTS) along a fiber-optic cable in the Brady Hot Springs observation well 56-1 (Aug 25, 2018). The cable samples temperature every ~25 cm along the borehole and every ~1 minute over ~20 hours, giving a roughly 2054 × 1210 grid (down-going leg only).

Goal: fit a 2D EFGP regression with a product squared-exponential kernel and compare the **Kronecker** vs **Jacobi** preconditioners for the mean CG solve.

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch

sys.path.append('..')
from efgpnd import EFGPND
from kernels import SquaredExponential
from load_brady import load_brady_grid, load_brady_torch, detrend_depth_polynomial

## 1. Load and inspect the raw temperature field

In [ ]:
z, t, T_raw, meta = load_brady_grid(downgoing_only=True)
print('shape (n_z, n_t):', T_raw.shape)
print('depth range:', z.min(), 'to', z.max(), 'm')
print('time range:', t.min(), 'to', t.max(), 's =', t.max()/3600, 'hr')
print('temperature range:', np.nanmin(T_raw), 'to', np.nanmax(T_raw), 'C')
print('start datetime:', meta['start_datetime'])
print('z step:', meta['z_step_m'], 'm; t step:', meta['t_step_s'], 's')
print('NaN count:', np.isnan(T_raw).sum())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(T_raw, aspect='auto', origin='lower',
               extent=[t.min()/3600, t.max()/3600, z.min(), z.max()],
               cmap='inferno')
ax.set_xlabel('Time (hours since start)')
ax.set_ylabel('Along-fiber distance (m)')
ax.set_title('Brady DTS — raw temperature field (down-going leg)')
plt.colorbar(im, ax=ax, label='Temperature (°C)')
plt.tight_layout()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(z, T_raw[:, T_raw.shape[1]//2])
ax1.set_xlabel('Along-fiber distance (m)')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Depth profile at mid-time')
ax1.grid(alpha=0.3)

ax2.plot(t/3600, T_raw[T_raw.shape[0]//2, :])
ax2.set_xlabel('Time (hours)')
ax2.set_ylabel('Temperature (°C)')
ax2.set_title('Time series at mid-depth')
ax2.grid(alpha=0.3)
plt.tight_layout()

## 2. Detrend the geothermal gradient

Temperature rises sharply with depth (geothermal source at the borehole bottom). A stationary GP cannot model that long-wavelength trend, so we subtract a degree-3 polynomial in depth at every time step. What remains is the spatially/temporally correlated residual structure (fracture-scale fluid signatures, transient flow events) we want the GP to capture.

In [ ]:
T_det = detrend_depth_polynomial(z, T_raw, deg=3)
print('detrended residual range:', np.nanmin(T_det), 'to', np.nanmax(T_det), 'C')
print('detrended std:', np.nanstd(T_det), 'C')

fig, ax = plt.subplots(figsize=(12, 6))
vmax = np.nanpercentile(np.abs(T_det), 98)
im = ax.imshow(T_det, aspect='auto', origin='lower',
               extent=[t.min()/3600, t.max()/3600, z.min(), z.max()],
               cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xlabel('Time (hours since start)')
ax.set_ylabel('Along-fiber distance (m)')
ax.set_title('Detrended temperature residual (degree-3 polynomial in depth removed)')
plt.colorbar(im, ax=ax, label='Residual T (°C)')
plt.tight_layout()

## 3. Prepare GP regression inputs

Coordinates are normalized to $[0, 1]^2$ and targets are standardized. We hold out a random 10% test split for RMSE evaluation.

In [ ]:
x_full, y_full, _ = load_brady_torch(downgoing_only=True, detrend=True, detrend_deg=3)
print('total points:', x_full.shape[0])

x_min, x_max = x_full.min(dim=0).values, x_full.max(dim=0).values
x_full = (x_full - x_min) / (x_max - x_min)

y_mean, y_std = y_full.mean(), y_full.std()
y_full = (y_full - y_mean) / y_std

rng = np.random.default_rng(42)
n = x_full.shape[0]
perm = torch.from_numpy(rng.permutation(n))
n_test = n // 10
test_idx = perm[:n_test]
train_idx = perm[n_test:]

x_train = x_full[train_idx].to(dtype=torch.float32)
y_train = y_full[train_idx].to(dtype=torch.float32)
x_test = x_full[test_idx].to(dtype=torch.float32)
y_test = y_full[test_idx].to(dtype=torch.float32)

print('train:', x_train.shape, 'test:', x_test.shape)
print('y_train mean/std (post-norm):', float(y_train.mean()), float(y_train.std()))

## 4. Hyperparameter learning under Kronecker vs Jacobi

Single isotropic SE kernel (we ignore per-dimension lengthscales for now). Each Adam step calls `model.compute_gradients()`, which performs a mean CG solve and a stochastic-trace CG solve under the chosen preconditioner. We record the hyperparameter trajectory, CG iteration counts, and per-step wall time. Both preconditioners should converge to the same hypers — only the per-step CG cost differs.

In [ ]:
from torch.optim import Adam

EPS = 1e-3
CG_TOL = 1e-4
NOISE_FLOOR = 1e-4
INIT_LENGTHSCALE = 0.1
INIT_VARIANCE = 1.0
INIT_NOISE = 0.5
LR = 0.1
MAX_ITERS = 30
J = 1  # Hutchinson trace samples
d = x_train.shape[1]


def run_training(precond_type, max_iters=MAX_ITERS, verbose_every=5):
    kernel = SquaredExponential(
        dimension=d,
        init_lengthscale=INIT_LENGTHSCALE,
        init_variance=INIT_VARIANCE,
    )
    model = EFGPND(
        x_train, y_train,
        kernel=kernel,
        sigmasq=INIT_NOISE,
        eps=EPS,
        estimate_params=False,
        opts={
            'mean_cg_preconditioner': True,
            'trace_cg_preconditioner': True,
            'mean_cg_preconditioner_type': precond_type,
            'mean_cg_warm_start': True,
            'cg_tolerance': CG_TOL,
        },
    )
    optimizer = Adam(model.parameters(), lr=LR)

    log = {
        'iter': [0],
        'lengthscale': [model.kernel.get_hyper('lengthscale')],
        'variance':    [model.kernel.get_hyper('variance')],
        'sigmasq':     [model._gp_params.sig2.item()],
        'mean_cg_iters':  [None],
        'trace_cg_iters': [None],
        'mtot':           [None],
        'feature_count':  [None],
        'step_sec':       [None],
    }

    for it in range(max_iters):
        optimizer.zero_grad()
        t0 = time.time()
        model.compute_gradients(trace_samples=J, cg_tol=CG_TOL, noise_floor=NOISE_FLOOR)
        optimizer.step()
        step_sec = time.time() - t0

        s = model.last_gradient_stats
        log['iter'].append(it + 1)
        log['lengthscale'].append(model.kernel.get_hyper('lengthscale'))
        log['variance'].append(model.kernel.get_hyper('variance'))
        log['sigmasq'].append(model._gp_params.sig2.item())
        log['mean_cg_iters'].append(s.get('mean_cg_iters'))
        log['trace_cg_iters'].append(s.get('trace_cg_iters'))
        log['mtot'].append(s.get('mtot'))
        log['feature_count'].append(s.get('feature_count'))
        log['step_sec'].append(step_sec)

        if it % verbose_every == 0 or it == max_iters - 1:
            print(f"  iter {it+1:>3}  ℓ={log['lengthscale'][-1]:.4g}  "
                  f"σ_f²={log['variance'][-1]:.4g}  σ_n²={log['sigmasq'][-1]:.4g}  "
                  f"cg(mean/trace)={s.get('mean_cg_iters')}/{s.get('trace_cg_iters')}  "
                  f"M={s.get('feature_count')}  t={step_sec:.2f}s")

    pred = model.predict(x_new=x_test, return_variance=False)
    pred = pred[0] if isinstance(pred, tuple) else pred
    rmse = float(torch.sqrt(((pred - y_test) ** 2).mean()))
    log['test_rmse'] = rmse
    log['model'] = model
    return log

In [ ]:
torch.manual_seed(0)
results = {}
for precond in ['kronecker', 'jacobi']:
    print(f'=== preconditioner: {precond} ===')
    results[precond] = run_training(precond)
    print(f"  test RMSE (standardized): {results[precond]['test_rmse']:.5f}")
    total = sum(s for s in results[precond]['step_sec'] if s is not None)
    print(f"  total training time: {total:.1f}s\n")

print('Summary:')
print(f"{'precond':<12} {'final ℓ':>10} {'final σ_f²':>11} {'final σ_n²':>11} "
      f"{'mean cg sum':>12} {'trace cg sum':>13} {'wall (s)':>10} {'rmse':>10}")
for precond, r in results.items():
    sum_mean  = sum(c for c in r['mean_cg_iters'] if c is not None)
    sum_trace = sum(c for c in r['trace_cg_iters'] if c is not None)
    wall = sum(s for s in r['step_sec'] if s is not None)
    print(f"{precond:<12} {r['lengthscale'][-1]:>10.4g} {r['variance'][-1]:>11.4g} "
          f"{r['sigmasq'][-1]:>11.4g} {sum_mean:>12d} {sum_trace:>13d} {wall:>10.2f} "
          f"{r['test_rmse']:>10.5f}")

In [ ]:
colors = {'kronecker': 'tab:blue', 'jacobi': 'tab:orange'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for precond, r in results.items():
    c = colors[precond]
    axes[0].plot(r['iter'], r['lengthscale'], 'o-', color=c, label=precond)
    axes[1].plot(r['iter'], r['variance'],    'o-', color=c, label=precond)
    axes[2].plot(r['iter'], r['sigmasq'],     'o-', color=c, label=precond)

for ax, title, ylabel in zip(
    axes,
    ['Lengthscale ℓ', 'Signal variance σ_f²', 'Noise variance σ_n²'],
    ['ℓ', 'σ_f²', 'σ_n²'],
):
    ax.set_xlabel('Adam iteration')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_yscale('log')
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle('Hyperparameter trajectories — should match across preconditioners')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
iters_plot = lambda log: log['iter'][1:]   # skip iter 0 (no CG yet)

for precond, r in results.items():
    c = colors[precond]
    axes[0].plot(iters_plot(r), r['mean_cg_iters'][1:],  'o-', color=c, label=precond)
    axes[1].plot(iters_plot(r), r['trace_cg_iters'][1:], 'o-', color=c, label=precond)
    axes[2].plot(iters_plot(r), r['step_sec'][1:],       'o-', color=c, label=precond)

axes[0].set_title('Mean CG iterations per Adam step')
axes[0].set_ylabel('CG iters')
axes[1].set_title('Trace CG iterations per Adam step')
axes[1].set_ylabel('CG iters')
axes[2].set_title('Wall time per Adam step')
axes[2].set_ylabel('seconds')
for ax in axes:
    ax.set_xlabel('Adam iteration')
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle('Per-step CG cost — where the preconditioners actually differ')
plt.tight_layout()

## 5. Posterior mean reconstruction (trained Kronecker model)

In [ ]:
model = results['kronecker']['model']

n_z_pred, n_t_pred = 256, 256
g_z = torch.linspace(0, 1, n_z_pred, dtype=x_train.dtype)
g_t = torch.linspace(0, 1, n_t_pred, dtype=x_train.dtype)
GZ, GT = torch.meshgrid(g_z, g_t, indexing='ij')
x_grid = torch.stack([GZ.flatten(), GT.flatten()], dim=1)

mean_pred = model.predict(x_new=x_grid, return_variance=False)
mean_pred = mean_pred[0] if isinstance(mean_pred, tuple) else mean_pred
mean_grid = (mean_pred.detach().cpu() * y_std + y_mean).numpy().reshape(n_z_pred, n_t_pred)

fig, ax = plt.subplots(figsize=(12, 6))
vmax = np.nanpercentile(np.abs(mean_grid), 98)
im = ax.imshow(mean_grid, aspect='auto', origin='lower',
               extent=[t.min()/3600, t.max()/3600, z.min(), z.max()],
               cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xlabel('Time (hours since start)')
ax.set_ylabel('Along-fiber distance (m)')
ax.set_title(
    f"EFGP posterior mean of detrended residual — "
    f"ℓ={model.kernel.get_hyper('lengthscale'):.3g}, "
    f"σ_f²={model.kernel.get_hyper('variance'):.3g}, "
    f"σ_n²={model._gp_params.sig2.item():.3g}"
)
plt.colorbar(im, ax=ax, label='Residual T (°C)')
plt.tight_layout()